In [2]:
from langgraph.graph import StateGraph, START
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
import requests

In [3]:
load_dotenv()
llm=ChatOpenAI(proxy_model_name='gpt-4o')


True

In [4]:
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA')
    using Alpha Vantage with API key in the URL.
    """
    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    )
    r = requests.get(url)
    return r.json()


In [ ]:
@tool
def purchase_stock(symbol:str, quantity:int) -> dict:
    """
    Simulate purchasing a given quantity of a stock symbol.
    HUMAN-IN-THE-LOOP: Interrupts and waits for human approval.
    """
    decision = interrupt(f"Approve buying {quantity} shares of {symbol}? (yes/no)")
    if isinstance(decision, str) and decision.lower() == "yes":
        return {
            "status": "success",
            "message": f"Purchase order placed for {quantity} shares of {symbol}.",
            "symbol": symbol,
            "quantity": quantity,
        }
    else:
        return {
            "status": "cancelled",
            "message": f"Purchase of {quantity} shares of {symbol} was declined by human.",
            "symbol": symbol,
            "quantity": quantity,
        }


In [ ]:
# --- State ---
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# --- Bind tools to LLM ---
tools = [get_stock_price, purchase_stock]
llm_with_tools = llm.bind_tools(tools)

# --- chat_node ---
def chat_node(state: ChatState):
    messages = state["messages"]
    if not messages:
        return {"messages": []}
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# --- Graph ---
tool_node = ToolNode(tools)
checkpointer = MemorySaver()

graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

app = graph.compile(checkpointer=checkpointer)
app


In [ ]:
# --- Run ---
config = {"configurable": {"thread_id": "thread-1"}}

result = app.invoke(
    {"messages": [HumanMessage(content="What is the stock price of AAPL?")]},
    config=config
)
print(result["messages"][-1].content)


In [ ]:
# --- Resume after human approval (run after interrupt) ---
# Change to "no" to cancel
result = app.invoke(Command(resume="yes"), config=config)
print(result["messages"][-1].content)
